# Entrenamiento del modelo de residuos (YOLO11n)
**NOTA: Este archivo se debe ejecutar dentro de google colab**

Ejecuta las celdas **en orden**, de arriba hacia abajo. Antes de empezar asegurate de lo siguiente:

1. Menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*.
2. En el panel izquierdo crea un secreto (el icono de llave) llamado `ROBOFLOW_API_KEY` con la clave API de Roboflow y activa *Acceso desde el notebook*. **Nunca** pegues la clave en el código.
3. En Roboflow, genera una **versión** del dataset (*Versions → Create New Version*), sin augmentations por ahora: Ultralytics ya aplica las suyas al entrenar.

## 1. Comprobar GPU

In [ ]:
!nvidia-smi

## 2. Instalar librerías
Necesitamos la libreria de robo para descargar el dataset y la libreria ultra para el entrenamiento

In [ ]:
!pip install -q ultralytics roboflow

## 3. Configuración inicial


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

REPO_URL = "https://github.com/FakeJohnG/Proyect_trashy.git"   # Repo del proyecto
ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
WORKSPACE = "juan-delgado6-utp-ac-pa"     # Workspace de Roboflow
PROYECTO = "trashy-bglwg"                 # nombre (id) del proyecto en Roboflow
VERSION = 1                               # número de versión del dataset 
RUN_NAME = "v1"                           # nombre de este entrenamiento
RUNS_DIR = "/content/drive/MyDrive/Trashy/runs"
MODELOS_DIR = "/content/drive/MyDrive/Trashy/modelos"

## 4. Traer el código del repositorio
Si ya está clonado, solo actualiza con `git pull`.

In [ ]:
!git clone {REPO_URL} /content/residuos-ml 2>/dev/null || git -C /content/residuos-ml pull


Una ves clonado cambia el directorio a la carpeta de content

In [ ]:
%cd /content/residuos-ml

## 5. Descargar el dataset desde Roboflow

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
version = rf.workspace(WORKSPACE).project(PROYECTO).version(VERSION)
dataset = version.download("yolov11")   # si da error, prueba "yolov8": el formato de etiquetas es el mismo
DATASET_DIR = dataset.location
print("Dataset descargado en:", DATASET_DIR)

## 6. Agrupar las 13 subclases en las 7 clases finales
Usa `scripts/agrupar_clases.py` y `configs/mapeo_clases.json` del repo. Si el dataset trae una subclase que no está en el mapeo, se detiene con un aviso en vez de etiquetar mal en silencio. Imprime cuántas cajas hay por clase: si alguna tiene muy pocas, hay que tomar más fotos de esa.

In [ ]:
!python scripts/agrupar_clases.py --src "{DATASET_DIR}" --dst /content/residuos_v1
print("Clases agrupadas en:", "/content/residuos_v1")
print("Listo para entrenar.")

## 7. Entrenar
- `yolo11n.pt`: pesos preentrenados (transfer learning), no se parte de cero.
- `epochs`: vueltas completas al dataset. `patience=20`: si no mejora en 20 épocas, se detiene solo.
- `imgsz=640`: tamaño al que se redimensionan las imágenes. `batch=16`: imágenes por paso (si da error de memoria, baja a 8).

In [ ]:
from pathlib import Path
from ultralytics import YOLO

DATA_YAML = Path("/content/residuos_v1/data_v1.yaml")

model = YOLO("yolo11n.pt")
resultados = model.train(
    data=str(DATA_YAML),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    project=RUNS_DIR,
    name=RUN_NAME,
)
RUN_DIR = Path(resultados.save_dir)
print("Resultados en:", RUN_DIR)

### (Solo si Colab se desconectó) Reanudar
1. Vuelve a ejecutar las celdas 2 a 6 (el dataset en `/content` se pierde al desconectarse).
2. Ajusta `RUN_DIR` a la carpeta de tu entrenamiento en Drive y ejecuta esta celda quitando los `#`.

In [ ]:
# from pathlib import Path
# from ultralytics import YOLO
# DATA_YAML = Path("/content/residuos_v1/data_v1.yaml")
# RUN_DIR = Path(RUNS_DIR) / RUN_NAME
# model = YOLO(RUN_DIR / "weights" / "last.pt")
# model.train(resume=True)

## 8. Evaluar
`best.pt` es la mejor época según la validación (`last.pt` es la última). Mira sobre todo la **matriz de confusión**: muestra qué materiales se confunden entre sí.

In [ ]:
from IPython.display import Image, display

best = YOLO(RUN_DIR / "weights" / "best.pt")
metricas = best.val(data=str(DATA_YAML))
print("mAP50:", round(metricas.box.map50, 3), "| mAP50-95:", round(metricas.box.map, 3))
display(Image(RUN_DIR / "confusion_matrix_normalized.png"))

## 9. Probar con fotos reales
Sube fotos tomadas con el celular (no del dataset) y mira qué predice. Es la prueba más honesta.

In [ ]:
from google.colab import files

subidas = files.upload()   # elige una o varias fotos
rutas = list(subidas.keys())
preds = best.predict(rutas, save=True, conf=0.25, project=str(RUN_DIR), name="pruebas", exist_ok=True)
for r in preds:
    display(Image(Path(r.save_dir) / Path(r.path).name))

## 10. Exportar modelo


Para web:

In [ ]:
ruta_tfjs = best.export(format="tfjs", imgsz=640)
print("Modelo web exportado:", ruta_tfjs)

Para android:

In [ ]:
ruta_tflite = best.export(format="tflite", int8=True, data=str(DATA_YAML), imgsz=640)
print("Modelo android exportado:", ruta_tflite)

## 11. Guardar el modelo terminado en Drive
Copia `best.pt`, el `.tflite` y la lista de clases (`clases.yaml`) a `MODELOS_DIR`. Los modelos **no** van a git: ver `models/README.md`.

In [ ]:
import shutil

destino = Path(MODELOS_DIR) / RUN_NAME
destino.mkdir(parents=True, exist_ok=True)
shutil.copy(RUN_DIR / "weights" / "best.pt", destino / "best.pt")
shutil.copy(DATA_YAML, destino / "clases.yaml")
for f in (RUN_DIR / "weights").rglob("*.tflite"):
    shutil.copy(f, destino / f.name)

for carpeta in (RUN_DIR / "weights").glob("*_web_model"):
    shutil.copytree(carpeta, destino / carpeta.name, dirs_exist_ok=True)


print("Guardado en:", destino)
print(sorted(p.name for p in destino.iterdir()))